In [3]:
from dataclasses import dataclass
from pathlib import Path

from agents.personal_assistant_agent import personal_assistant_agent
from configs import SKILLS_ROUTE_PREFIX
from configs.personal_assistant_config import DEFAULT_USER_ID, MEMORIES_ROUTE_PREFIX
from utils.memory_backend_util import make_skills_namespace

"""Virtual memory backend utility for DeepAgents and LangGraph Store routing."""

from collections.abc import Callable
from typing import Any

from deepagents.backends import (
    BackendProtocol,
    CompositeBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore

from configs.personal_assistant_config import (
    DEFAULT_USER_ID,
    MEMORIES_ROUTE_PREFIX,
)


@dataclass
class RoutePrefix:
    memories: str|None = MEMORIES_ROUTE_PREFIX
    skills: str|None = "skills"



def make_memory_namespace(prefix_reg: RoutePrefix, runtime: Any | None = None) -> tuple[str, str]:
    """Resolve the per-user memory namespace tuple from the LangGraph execution runtime.

    Under the hood, DeepAgents StoreBackend executes this callable during file operations
    to determine which store partition (namespace) to write to or read from.

    If runtime is None (e.g. backend accessed outside of graph invoke) or configurable
    does not specify a user_id, falls back to DEFAULT_USER_ID.
    """
    user_id = DEFAULT_USER_ID
    memory_prefix = prefix_reg.memories
    if runtime is not None:
        config = getattr(runtime, "config", {}) or {}
        configurable = config.get("configurable", {})
        candidate = configurable.get("user_id")
        if candidate:
            user_id = str(candidate).strip()

    prefix = memory_prefix
    return user_id, prefix

def make_skills_namespace(runtime):
    return ("skills",)


def make_memory_composite_backend(
        store: BaseStore | None = None,
        default_backend: BackendProtocol | None = None,
        namespace_factory: Callable[[Any | None], tuple[str, str]] = make_memory_namespace,
) -> CompositeBackend:
    """Build a CompositeBackend routing persistent /memories/ paths to StoreBackend.

    - Default backend (unmatched paths, e.g. /workspace/, /temp.txt): StateBackend (ephemeral
      to the current conversation thread).
    - Routed path (/memories/): StoreBackend wired to the persistent LangGraph BaseStore
      scoped dynamically per-user via `namespace_factory`.

    Args:
        store: LangGraph BaseStore instance (e.g. InMemoryStore or SqliteStore). If None,
            defaults to an InMemoryStore for Phase A testing.
        default_backend: Fallback backend for non-memory paths. Defaults to StateBackend().
        namespace_factory: Callable receiving runtime and returning the namespace tuple.
            Defaults to `make_memory_namespace`.

    Returns:
        A ready-to-use CompositeBackend instance.
    """
    if store is None:
        store = InMemoryStore()

    if default_backend is None:
        default_backend = StateBackend()

    return CompositeBackend(
        default=default_backend,
        routes={
            MEMORIES_ROUTE_PREFIX: StoreBackend(
                namespace=namespace_factory,
                store=store,
            ),
            SKILLS_ROUTE_PREFIX: StoreBackend(
                namespace=make_skills_namespace,
                store=store
            )
        },
    )


In [ ]:
DEFAULT_SKILL_DIR = Path(__file__)/"folanja"
def bootstrap_skills(backend: BackendProtocol, skill_dir: Path = DEFAULT_SKILL_DIR):
    
